# Practical Exerecise: Create an R-Tree structure to import data points

### Reuqirements:
- Suppose “B=4”, you need to consider the lower bound and upper bound in this exercise.
- A function “Choose Subtree” should be devised to decide where should we insert a new data point.
- A function “Handel Overflow” should be devised, where the perimeter sum should be considered in splitting a node or an MBR.
- In the splitting, we need to consider the perimeter sum of several combinations of the splits

### Proerties of the R-Tree:
1. B=4, each node in the tree has at least 2 childs and at most 4 childs
2. Create a "Node" class to save MBR and nodes in each rectangle; 
3. Create a "RTree" class to store all rectangles based on the "Node" class.
4. Define “Choose Subtree” function to decide where should we insert a new data point.
5. Define “Handel Overflow” function to handle overflow situation. The perimeter sum should be considered in splitting a node or an MBR

### 1. Define Node class

### functions of the node class:
1. calculate perimeter of the MBR of the node
2. check whether the node is overflow
3. check whether the node is the root node
4. check whether the node is a leaf node

In [1]:
B = 4
class Node(object): 
    def __init__(self):
        self.id = 0
        self.child_nodes = [] #for non-leaf node, a list to store nodes
        self.data_points = [] #for leaf node, a list to store data points
        self.parent = None
        self.MBR = {
            'x1': -1,
            'y1': -1,
            'x2': -1,
            'y2': -1,
        }
    def perimeter(self):
        return #perimeter of the MBR

    def is_overflow(self):
        return #return True if the node is overflow, return False if the node is not overflow

    def is_root(self):
        return #return True if the node is a root node, return False if the node is not a root node

    def is_leaf(self):
        return #return True if the node is a leaf node, return False if the node is not a leaf node

### 2. Define RTree class

### functions of the RTree class:
1. insert function to insert node into the R-tree
2. handle_overflow function to deal with overflow

In [ ]:
import sys
import math

class RTree(object): #R tree class
    def __init__(self):
        self.root = Node() #Create a root

    def insert(self, node, point): # insert p(data point) to u (MBR)
        pass

    def add_data_point(self, node, data_point): #add data points and update the the MBRS
        # 1. add the point into node.data_points
        # 2. update node.MBR
        node.data_points.append(data_point)
        if data_point['x'] < node.MBR['x1']:
            node.MBR['x1'] = data_point['x']
        if data_point['x'] > node.MBR['x2']:
            node.MBR['x2'] = data_point['x']
        if data_point['y'] < node.MBR['y1']:
            node.MBR['y1'] = data_point['y']
        if data_point['y'] > node.MBR['y2']:
            node.MBR['y2'] = data_point['y']

    def add_child(self, node, child):
        # 1. add the child into node.child_nodes
        # 2. set the node as the parent of the child
        # 3. update node.MBR
        node.child_nodes.append(child) #add child nodes to the current parent (node) and update the MBRs. It is used in handeling overflows
        child.parent = node
        if child.MBR['x1'] < node.MBR['x1']:
            node.MBR['x1'] = child.MBR['x1']
        if child.MBR['x2'] > node.MBR['x2']:
            node.MBR['x2'] = child.MBR['x2']
        if child.MBR['y1'] < node.MBR['y1']:
            node.MBR['y1'] = child.MBR['y1']
        if child.MBR['y2'] > node.MBR['y2']:
            node.MBR['y2'] = child.MBR['y2']

    # return the child whose MBR requires the minimum increase in perimeter to cover p

    def handle_overflow(self, node):
        node1, node2 = self.split(node) #u1 u2 are the two splits returned by the function "split"
        # if u is root, create a new root with s1 and s2 as its' children
        if node.is_root():
            new_root = Node()
            self.add_child(new_root, node1)
            self.add_child(new_root, node2)
            self.root = new_root
            self.update_mbr(new_root)
        # if u is not root, delete u, and set s1 and s2 as u's parent's new children
        else:
            parent = node.parent
            # copy the information of s1 into u
            parent.child_nodes.remove(node)
            self.add_child(parent, node1) #link the two splits and update the corresponding MBR
            self.add_child(parent, node2)
            if parent.is_overflow(): #check the parent node recursively
                self.handle_overflow(parent)


    def choose_subtree(self, node, point): 
        if node.is_leaf(): #find the leaf and insert the data point
            return node
        else:
            min_increase = sys.maxsize #set an initial value
            best_child = None
            for child in node.child_nodes: #check each child to find the best node to insert the point 
                if min_increase > self.peri_increase(child, point):
                    min_increase = self.peri_increase(child, point)
                    best_child = child
            return best_child

    def peri_increase(self, node, point): # calculate the increase of the perimeter after inserting the new data point
        # new perimeter - original perimeter = increase of perimeter
        origin_mbr = node.MBR
        x1, x2, y1, y2 = origin_mbr['x1'], origin_mbr['x2'], origin_mbr['y1'], origin_mbr['y2']
        increase = (max([x1, x2, point['x']]) - min([x1, x2, point['x']]) +
                    max([y1, y2, point['y']]) - min([y1, y2, point['y']])) - node.perimeter()
        return increase
            
    def split(self, u):
        # split u into s1 and s2
        best_s1 = Node()
        best_s2 = Node()
        best_perimeter = sys.maxsize
        # u is a leaf node
        if u.is_leaf():
            m = u.data_points.__len__()
            # create two different kinds of divides
            divides = [sorted(u.data_points, key=lambda data_point: data_point['x']),
                       sorted(u.data_points, key=lambda data_point: data_point['y'])] #sorting the points based on X dimension and Y dimension
            for divide in divides:
                for i in range(math.ceil(0.4 * B), m - math.ceil(0.4 * B) + 1): #check the combinations to find a near-optimal one
                    s1 = Node()
                    s1.data_points = divide[0: i]
                    self.update_mbr(s1)
                    s2 = Node()
                    s2.data_points = divide[i: divide.__len__()]
                    self.update_mbr(s2)
                    if best_perimeter > s1.perimeter() + s2.perimeter(): 
                        best_perimeter = s1.perimeter() + s2.perimeter()
                        best_s1 = s1
                        best_s2 = s2

        # u is a internal node
        else:
            # create four different kinds of divides
            m = u.child_nodes.__len__()
            divides = [sorted(u.child_nodes, key=lambda child_node: child_node.MBR['x1']), #sorting based on MBRs
                       sorted(u.child_nodes, key=lambda child_node: child_node.MBR['x2']),
                       sorted(u.child_nodes, key=lambda child_node: child_node.MBR['y1']),
                       sorted(u.child_nodes, key=lambda child_node: child_node.MBR['y2'])]
            for divide in divides:
                for i in range(math.ceil(0.4 * B), m - math.ceil(0.4 * B) + 1): #check the combinations
                    s1 = Node()
                    s1.child_nodes = divide[0: i]
                    self.update_mbr(s1)
                    s2 = Node()
                    s2.child_nodes = divide[i: divide.__len__()]
                    self.update_mbr(s2)
                    if best_perimeter > s1.perimeter() + s2.perimeter():
                        best_perimeter = s1.perimeter() + s2.perimeter()
                        best_s1 = s1
                        best_s2 = s2

        for child in best_s1.child_nodes:
            child.parent = best_s1
        for child in best_s2.child_nodes:
            child.parent = best_s2

        return best_s1, best_s2

        
    def update_mbr(self, node): #update MBRs when forming a new MBR. It is used in checking the combinations and update the root
        x_list = []
        y_list = []
        if node.is_leaf():
            x_list = [point['x'] for point in node.data_points]
            y_list = [point['y'] for point in node.data_points]
        else:
            x_list = [child.MBR['x1'] for child in node.child_nodes] + [child.MBR['x2'] for child in node.child_nodes]
            y_list = [child.MBR['y1'] for child in node.child_nodes] + [child.MBR['y2'] for child in node.child_nodes]
        new_mbr = {
            'x1': min(x_list),
            'x2': max(x_list),
            'y1': min(y_list),
            'y2': max(y_list)
        }
        node.MBR = new_mbr

### 3. Import data points into the RTree
1. Read the file and save datapoints
2. build R-Tree and import the datapoints into the tree